## 根据序列获取MSA结果（使用mmseqs2的web服务器）

#### 将PDB 中根据序列seqs-》MSA ，使用理想的序列（即包含缺失残基）（confdiff中使用）进行MSA而不是PDB中存在的残基序列进行

复合物 (Complex) 预测的 MSA 生成
在 蛋白质复合物结构预测 中，MSA（多序列比对）分为 未配对（Unpaired MSA） 和 配对（Paired MSA） 两种。

未配对 MSA（Unpaired MSA）
生成方式与单个蛋白质结构预测相同。
通过搜索 UniRef100 和 环境序列数据库（environmental sequences）。
迭代次数： 每个数据库执行 3 次迭代。

配对 MSA（Paired MSA）
通过搜索 UniRef100 数据库获得序列比对。
配对策略：
同一物种或亚种（基于 NCBI 分类标识符，即 taxonomical identifier）。
仅当所有查询序列（即复合物的所有蛋白）在该分类标识符中都存在时，才会配对，否则默认使用未配对 MSA。

In [3]:
import pandas as pd
import sys
import os
sys.path.append("/opt/data/private/lyb/protenix")
from runner.msa_search import msa_search


将pkl转化为json文件

In [ ]:
# import pickle
# import json

# # 1. 读取 .pkl 文件（假设是 DataFrame 格式）
# pkl_path = '/mnt/public2/zhangyx/Projects/related_work/RFdiffusion/notebooks_script/confdiff_need/md_data_af3/md_data_no_repr.pkl'  # 替换为你的路径
# with open(pkl_path, 'rb') as f:
#     df = pickle.load(f)

# # 2. 转换格式：每行保存为 {'PDB_ID': ..., 'Seqs': [...]}
# result = []
# for _, row in df.iterrows():
#     entry = {
#         'PDB_ID': row['PDB_ID'],
#         'Seqs': row['seqs']  # 注意列名大小写要匹配
#     }
#     result.append(entry)

# # 3. 保存为 JSON 文件
# json_path = '/mnt/public2/zhangyx/Projects/related_work/RFdiffusion/notebooks_script/confdiff_need/md_data_af3/need_msa_output.json'
# with open(json_path, 'w') as f:
#     json.dump(result, f, indent=4)

# print(f"Saved {len(result)} entries to {json_path}")


In [2]:
import pandas as pd
data=pd.read_pickle("/opt/data/private/lyb/test/competition_dir/final_output_final_merged.pkl")
data

,PDB_ID,model,chain_ids,seqs,chain_lens,total_seqLens,entities_to_chains
0,9YDF,0,[A],[MHHHHHHSSGRENLYFQGPLTVEEVLYHIPWLDKNKDYINFIQEK...,[173],173,[{'1': ['A']}]
1,9U4S,0,"[A, B]",[GHFVQQKVKVFRAADPLVGVFLWGVAHSINELSQVPPPVMLLPDD...,"[349, 349]",698,"[{'1': ['A', 'B']}]"
2,9V80,0,"[A, B, C]",[MASERPEPEVEEAGQVFLLMKKDYRISRNVRLAWFLSHLHQTVQA...,"[3432, 447, 436]",4315,"[{'1': ['A']}, {'2': ['B']}, {'3': ['C']}]"


## 选择1：使用pkl文件获取MSA结果

In [6]:
# 第一步产生的pkl
pkl_dir="/opt/data/private/lyb/test/competition_dir/final_output_final_merged.pkl"
# msa 输出的文件夹
out_dir="/opt/data/private/lyb/test/competition_dir/msa_dir"

In [7]:
import os
from concurrent.futures import ProcessPoolExecutor
import pandas as pd

def process_row(row_dict, out_dir, log_file):
    seq_set=set(row_dict['seqs'])
    protein_seqs = sorted(seq_set)
    out_dirname = os.path.join(out_dir, str(row_dict['PDB_ID']))
    os.makedirs(out_dirname, exist_ok=True)
    print(f"Processing {row_dict['PDB_ID']} in {out_dirname}")
     # 将输出重定向到日志文件
    with open(log_file, "a") as log:
        log.write(f"Processing {row_dict['PDB_ID']} in {out_dirname}\n")
        # 将标准输出和标准错误输出都重定向到 log_file
        sys.stdout = log
        sys.stderr = log

        try:
            msa_res_subdirs = msa_search(protein_seqs, out_dirname)
            log.write(f"Completed: {row_dict['PDB_ID']}\n")
        except Exception as e:
            log.write(f"Error processing {row_dict['PDB_ID']}: {e}\n")
        finally:
            # 恢复stdout和stderr
            sys.stdout = sys.__stdout__
            sys.stderr = sys.__stderr__
    return row_dict['PDB_ID'], msa_res_subdirs

# 主函数
if __name__ == "__main__":
    pkl_df = pd.read_pickle(pkl_dir)
    df = pkl_df
    #以下不动
    # 清空日志文件
    log_file="/opt/data/private/lyb/test/data_test/msa_log.txt"
    open(log_file, "w").close()
    # 多进程执行
    with ProcessPoolExecutor(max_workers=40) as executor:
        # 提交任务
        futures = [
            executor.submit(process_row, row._asdict(), out_dir, log_file)
            for row in df.itertuples(index=False, name='Pandas')
        ]
        print("All tasks submitted.")
        # 收集结果
        for future in futures:
            try:
                pdb_id, msa_result = future.result()
                print(f"Completed: {pdb_id}")
            except Exception as e:
                with open(log_file, "a") as log:
                    log.write(f"Error processing row: {e}\n")


Processing 9YDF in /opt/data/private/lyb/test/competition_dir/msa_dir/9YDF
Processing 9U4S in /opt/data/private/lyb/test/competition_dir/msa_dir/9U4S


2025-11-14 12:56:34,952 [/opt/data/private/lyb/protenix/protenix/web_service/colab_request_utils.py:209] INFO protenix.web_service.colab_request_utils: Msa server is running.


Processing 9V80 in /opt/data/private/lyb/test/competition_dir/msa_dir/9V80

2025-11-14 12:56:34,961 [/opt/data/private/lyb/protenix/protenix/web_service/colab_request_utils.py:209] INFO protenix.web_service.colab_request_utils: Msa server is running.


2025-11-14 12:56:34,975 [/opt/data/private/lyb/protenix/protenix/web_service/colab_request_utils.py:209] INFO protenix.web_service.colab_request_utils: Msa server is running.


All tasks submitted.
Completed: 9YDF
Completed: 9U4S
Completed: 9V80


## 选择2：使用复合的fasta直接获取msa结果

In [12]:
from Bio import SeqIO
from collections import defaultdict

def parse_fasta(fasta_path):
    """解析 FASTA 文件，按 PDB ID（不含链信息）合并序列"""
    pdb_dict = defaultdict(list)  # 用于存储 PDB_ID 和对应的序列

    # 解析 FASTA 文件
    for record in SeqIO.parse(fasta_path, "fasta"):

        pdb_id, chain = record.id.split("_")  # 分离 PDB ID 和链

        # pdb_id= record.id
        pdb_id = pdb_id.upper()  # PDB ID 转换为大写
        pdb_dict[pdb_id].append(str(record.seq))  # 追加序列

    paired_count = sum(1 for v in pdb_dict.values() if len(v) > 1)  # 统计配对的数量
    unpaired_count = sum(1 for v in pdb_dict.values() if len(v) == 1)  # 统计未配对的数量

    sequences = [{"PDB_ID": pdb_id, "Seqs": seqs} for pdb_id, seqs in pdb_dict.items()]

    print(f"配对的 PDB 记录数: {paired_count}")
    print(f"未配对的 PDB 记录数: {unpaired_count}")


    return sequences

# fasta_path='/home/tom/fsas/data_16t/6_mono_test/7-9/output_merge.fasta'
# sequences=parse_fasta(fasta_path)
# for i in sequences[:5]:
#     print(i)


In [13]:
import os
import sys
from concurrent.futures import ProcessPoolExecutor
import pandas as pd
from Bio import SeqIO
import sys
import os
sys.path.append("/opt/data/private/lyb/Protenix-main")
from runner.msa_search import msa_search

def process_row(row_dict, out_dir, log_file):
    """处理单个 PDB_ID 及其序列"""
    seq_set = set(row_dict['Seqs'])  # 处理多个序列，去重
    protein_seqs = sorted(seq_set)  # 排序以保持一致性
    out_dirname = os.path.join(out_dir, str(row_dict['PDB_ID']))  # 生成输出目录
    os.makedirs(out_dirname, exist_ok=True)  # 确保目录存在

    with open(log_file, "a") as log:
        original_stdout = sys.stdout  # 备份 stdout
        original_stderr = sys.stderr  # 备份 stderr
        sys.stdout = log
        sys.stderr = log

        try:
            msa_res_subdirs = msa_search(protein_seqs, out_dirname)  # 进行 MSA 搜索
        except Exception as e:
            log.write(f"Error processing {row_dict['PDB_ID']}: {e}\n")
            msa_res_subdirs = None  # 发生错误时返回 None
        finally:
            sys.stdout = original_stdout  # 恢复 stdout
            sys.stderr = original_stderr  # 恢复 stderr

    return row_dict['PDB_ID'], msa_res_subdirs



In [14]:
def get_processed_pdb_ids(out_dir):
    """获取已处理的 PDB_ID 列表（从 MSA_OUTPUT 子文件夹名提取）"""
    processed_pdb_ids = set()

    for folder_name in os.listdir(out_dir):
        folder_path = os.path.join(out_dir, folder_name)
        if os.path.isdir(folder_path):  # 确保是文件夹
            processed_pdb_ids.add(folder_name.upper())  # 统一转换为大写

    return processed_pdb_ids

In [15]:
# if __name__ == "__main__":
#     fasta_path = "/home/tom/fsas/data_16t/6_mono_test/7-9/output_merge.fasta"
#     out_dir = "/home/tom/fsas/data_16t/6_mono_test/7-9/output"
#     log_file = "/mnt/public2/zhangyx/Projects/related_work/RFdiffusion/notebooks_script/confdiff_need/temp_8rxa/msa_log.log"

#     # 解析 FASTA 文件
#     sequence_data = parse_fasta(fasta_path)
#     print("总共需要处理：",len(sequence_data))  
#     # 获取已处理的 PDB_ID
#     processed_pdb_ids = get_processed_pdb_ids(out_dir)
#     print("已处理：",len(processed_pdb_ids))        
#     # 过滤掉已处理的 PDB_ID
#     sequence_data = [entry for entry in sequence_data if entry["PDB_ID"] not in processed_pdb_ids]
#     print("现需要处理：",len(sequence_data))
#     # 清空日志文件
#     open(log_file, "w").close()

#     # 使用多进程并行执行
#     with ProcessPoolExecutor(max_workers=10) as executor:
#         futures = {
#             executor.submit(process_row, row, out_dir, log_file): row['PDB_ID']
#             for row in sequence_data
#         }

#         # 收集结果
#         for future in futures:
#             pdb_id = futures[future]  # 获取当前任务对应的 PDB_ID
#             try:
#                 pdb_id, msa_result = future.result()
#             except Exception as e:
#                 error_message = f" Error processing {pdb_id}: {e}\n"
#                 with open(log_file, "a") as log:
#                     log.write(error_message)


In [16]:
import os
import time

fasta_path = "/opt/data/private/lyb/test/data_test/rcsb_pdb_9V80.fasta"
out_dir = "/opt/data/private/lyb/test/data_test/msa_dir"
sequence_data = parse_fasta(fasta_path)
print("总共需要处理：",len(sequence_data))  
# 获取已处理的 PDB_ID
processed_pdb_ids = get_processed_pdb_ids(out_dir)
print("已处理：",len(processed_pdb_ids))        
# 过滤掉已处理的 PDB_ID
sequence_data = [entry for entry in sequence_data if entry["PDB_ID"]  not in processed_pdb_ids]
print("现需要处理：",len(sequence_data))

for i, row in enumerate(sequence_data):
    pdb_id = row["PDB_ID"]
    protein_seqs = sorted(row["Seqs"])
    out_dirname = os.path.join(out_dir, str(pdb_id))
    os.makedirs(out_dirname, exist_ok=True)
    msa_res_subdirs = msa_search(protein_seqs, out_dirname)  # 进行 MSA 搜索

    print(msa_res_subdirs)
    # 运行 MMseqs2 搜索
    print(f"Finished processing PDB_ID: {pdb_id}, sleeping for 1s...")
    time.sleep(1)

2025-11-13 14:52:47,309 [/opt/data/private/lyb/Protenix-main/protenix/web_service/colab_request_utils.py:209] INFO protenix.web_service.colab_request_utils: Msa server is running.


配对的 PDB 记录数: 1
未配对的 PDB 记录数: 0
总共需要处理： 1
已处理： 3
现需要处理： 1


SUBMIT:   0%|          | 0/100 [elapsed: 00:00 estimate remaining: ?]

RUNNING:  13%|█▎        | 13/100 [elapsed: 05:06 estimate remaining: 34:13]


KeyboardInterrupt: 

In [ ]:
import json
from pathlib import Path
json_path='/mnt/public2/zhangyx/Projects/related_work/RFdiffusion/notebooks_script/confdiff_need/md_data_af3/need_msa_output.json'
with open(json_path, 'r') as f:
    data = json.load(f)
num_chunk=5
# 计算每份的大小
chunk_size = len(data) // num_chunk + (len(data) % num_chunk > 0)
save_path=Path("/mnt/public2/zhangyx/Projects/related_work/RFdiffusion/notebooks_script/confdiff_need/md_data_af3/new_md_1800_msa")
# 拆分数据
print("数据已拆分并保存为:")
for i in range(num_chunk):
    chunk = data[i * chunk_size:(i + 1) * chunk_size]
    with open(save_path/f'split_part_{i+1}.json', 'w') as f:
        json.dump(chunk, f, indent=2)
    print(f"数据已拆分并保存为 split_part_{i+1}.json")



#### 获取AF3中断后，获取未处理的json文件集合

In [ ]:
import os
from collections import defaultdict

# 指定目标文件夹
folder_path = "/mnt/public2/zhangyx/Projects/related_work/ConfDiff/data/Af3_characterization/30000_40000"

# 统计 PDBid 出现次数
pdbid_count = defaultdict(int)

# 遍历文件夹中的所有文件
for file in os.listdir(folder_path):
    if file.endswith(".npy"):  # 只处理 .npy 文件
        pdbid = file.split("_")[0]  # 获取文件名前缀（PDBid）
        pdbid_count[pdbid] += 1  # 统计出现次数

# 分类 PDBid
passed_pdbids = [pdb for pdb, count in pdbid_count.items() if count == 2]
failed_pdbids = [pdb for pdb, count in pdbid_count.items() if count == 1]

# 输出结果
print("已有成对表征的 PDBid：", passed_pdbids)
print("未有成对表征的 PDBid：", failed_pdbids)
print("已有成对表征的 PDBid：", len(passed_pdbids))
print("未有成对表征的 PDBid：", len(failed_pdbids))


In [ ]:
import pandas as pd
pkl_filepath="/mnt/public2/zhangyx/Projects/related_work/RFdiffusion/notebooks_script/confdiff_need/msatest/myself_seqs_test/metadata.pkl"
df=pd.read_pickle(pkl_filepath)
df[df["PDB_ID"]=="1MHQ"]

In [ ]:
import os

directory = "/mnt/public2/zhangyx/Projects/related_work/ConfDiff/data/json_with_msa/10000-30000_177_json"
pdb_ids = set()

for filename in os.listdir(directory):
    if filename.endswith(".json"):
        pdb_ids.add(filename[:4])  # 获取前四个字符
# print(len(pdb_ids))
# need_again_processed_pdb_id=list(pdb_ids-set(passed_pdbids))
with open("/mnt/public2/zhangyx/Projects/related_work/ConfDiff/data/177_af3_pdb_ids.txt", "w") as f:
    f.write("\n".join(pdb_ids))

获取还没生成表征的pdbid

In [ ]:
file_path = '/mnt/public2/zhangyx/Projects/related_work/ConfDiff/data/json_with_msa/177_af3_40000_ag1_pdb_ids.txt'
pdb_ids=[]
# 读取文件并将每行的 pdbid 存入列表
with open(file_path, 'r') as file:
    pdb_ids = [line.strip() for line in file.readlines()]
need_again_processed_pdb_id=pdb_ids
print(len(need_again_processed_pdb_id))

In [ ]:
import os
import json

# 指定目标文件夹
folder_path = "/mnt/public2/zhangyx/Projects/related_work/ConfDiff/data/json_with_msa/40000-end_177_json"

# 初始化列表，用于存储拼接的内容
combined_list = []

# 遍历文件夹中的所有文件
for file_name in os.listdir(folder_path):
    pdbid=os.path.basename(file_name).split("_")[0]
    
    if (pdbid in need_again_processed_pdb_id) and (file_name.endswith(".json")):  # 筛选出 JSON 文件
        file_path = os.path.join(folder_path, file_name)  # 构建文件路径
        # print(file_path)
        with open(file_path, 'r') as file:  # 打开 JSON 文件
            data = json.load(file)  # 加载 JSON 数据
            # print(data)
            combined_list.append(data)  # 将数据添加到列表
print(len(combined_list))
merge_json_dir="/mnt/public2/zhangyx/Projects/related_work/ConfDiff/data/json_with_msa/"
merge_json_path=os.path.join(merge_json_dir,"40000-end_177_ag1_merge_json_msa.json")
with open(merge_json_path, "w") as json_file:
    json.dump(combined_list, json_file, indent=4)


文件批量重命名

In [ ]:
import os
import glob

# 指定目标文件夹
folder_path = "/mnt/lyb_data/lyb_af3_data/Af3_characterization/1000_test"

# 查找所有符合 *_pair_repr_recycle3.npy 格式的文件
files = glob.glob(os.path.join(folder_path, "*_single_repr_recycle3.npy"))
print(len(files))
# 遍历文件并重命名
for file in files:
    # 提取文件名（不包含路径）
    filename = os.path.basename(file)
    
    # 替换 _pair_repr_recycle3.npy 为 _pair_10.npy
    new_filename = filename.replace("_single_repr_recycle3.npy", "_single_10.npy")
    
    # 生成完整的新路径
    new_filepath = os.path.join(folder_path, new_filename)
    
    # 重命名文件
    os.rename(file, new_filepath)
    # print(f"重命名: {filename} -> {new_filename}")

